# SST Indices Preprocessing

This notebook runs the generalized SST index preprocessing workflow. It calls `scripts/run_process_sst_index.py` to:
1. Load raw monthly data for E3SM, observations (HadISST2), and CESM-SMYLE.
2. Compute monthly and seasonal averages for 13 SST indices (including Nino regions, TNA, TSA, IOD, TNI, ONI, RONI, and Atlantic indices).
3. Save the results as NetCDF files to the diagnostic output directories.

This allows the downstream diagnostics notebook `4_refactor_sst_skill_ts.ipynb` to load these precalculated indices instantly.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

# Identify repository root
REPO_ROOT = Path(os.getcwd())
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = Path("..")

In [2]:
# -----------------------------
# Configuration
# -----------------------------
CONFIG = {
    "sources": ["obs", "e3sm", "smyle"],
    "regions": [
        "IOD", "TNI", "ONI", "RONI",
        "Nino12", "Nino3", "Nino3.4", "Nino4",
        "TNA", "TSA", "PACWRAMPOOL", "AtlNino", "AtlMDR"
    ],  # List of target indices/regions to compute
    "custom_regions": {
        "Nino12": {
            "lonlat": [270.0, 280.0, -10.0, 0.0],
            "long_name": "Nino 1+2 regional mean SST",
        },
        "Nino3": {
            "lonlat": [210.0, 270.0, -5.0, 5.0],
            "long_name": "Nino 3 regional mean SST",
        },
        "Nino3.4": {
            "lonlat": [190.0, 240.0, -5.0, 5.0],
            "long_name": "Nino 3.4 regional mean SST",
        },
        "Nino4": {
            "lonlat": [160.0, 210.0, -5.0, 5.0],
            "long_name": "Nino 4 regional mean SST",
        },
        "TNA": {
            "lonlat": [305.0, 345.0, 5.0, 25.0],
            "long_name": "TNA regional mean SST",
        },
        "TSA": {
            "lonlat": [330.0, 10.0, -20.0, 0.0],
            "long_name": "TSA regional mean SST",
        },
        "PACWRAMPOOL": {
            "lonlat": [60.0, 170.0, -15.0, 15.0],
            "long_name": "PACWRAMPOOL regional mean SST",
        },
        "AtlNino": {
            "lonlat": [340.0, 360.0, -3.0, 3.0],
            "long_name": "Atlantic Nino regional mean SST",
        },
        "AtlMDR": {
            "lonlat": [280.0, 350.0, 10.0, 20.0],
            "long_name": "Atlantic MDR regional mean SST",
        },
        # Helper regions for derived indices
        "IOD_West": {
            "lonlat": [50.0, 70.0, -10.0, 10.0],
            "long_name": "IOD West regional mean SST",
        },
        "IOD_East": {
            "lonlat": [90.0, 110.0, -10.0, 0.0],
            "long_name": "IOD East regional mean SST",
        },
        "TropicalMean": {
            "lonlat": [0.0, 360.0, -20.0, 20.0],
            "long_name": "Tropical Mean regional mean SST",
        },
    },
    "outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE",
    "smyle_outdir": "/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE",
    "init_months": [5, 11],
    "year_start": 1980,
    "year_end": 2018,
    "climy0": 1980,
    "climy1": 2010,
    "nlead": 24,
    "e3sm_nens": 10,
    "smyle_nens": 20,
    "workers": 8,
    "force": True,  # Set to True to force rewrite
}

In [3]:
# -----------------------------
# Construct and execute command for each index in a loop
# -----------------------------
script_path = str(REPO_ROOT / "scripts" / "run_process_sst_index.py")

# Set environment variables for GDAL/PROJ
env = os.environ.copy()
conda_prefix = "/global/homes/z/zhan391/.conda/envs/e3sm_analysis"
env["GDAL_DATA"] = f"{conda_prefix}/share/gdal"
env["PROJ_LIB"] = f"{conda_prefix}/share/proj"

for r in CONFIG["regions"]:
    print("=" * 60)
    print(f"Processing SST index: {r}")
    print("=" * 60)

    cmd = [
        sys.executable,
        script_path,
        "--sources", *CONFIG["sources"],
        "--outdir", CONFIG["outdir"],
        "--smyle-outdir", CONFIG["smyle_outdir"],
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--climy0", str(CONFIG["climy0"]),
        "--climy1", str(CONFIG["climy1"]),
        "--nlead", str(CONFIG["nlead"]),
        "--e3sm-nens", str(CONFIG["e3sm_nens"]),
        "--smyle-nens", str(CONFIG["smyle_nens"]),
        "--workers", str(CONFIG["workers"]),
        "--regions", r,
    ]

    if CONFIG.get("custom_regions"):
        import json
        cmd.extend(["--custom-regions", json.dumps(CONFIG["custom_regions"])])

    if CONFIG["force"]:
        cmd.append("--force")

    print("Running command:")
    print(" ".join(cmd))

    # Run the preprocessing script
    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    print("\n--- STDOUT ---")
    print(result.stdout)

    if result.returncode != 0:
        print("\n--- STDERR ---")
        print(result.stderr)
        raise RuntimeError(f"Preprocessing failed for index '{r}' with exit code {result.returncode}")
        
print("\nAll SST indices processed successfully!")

Processing SST index: IOD
Running command:
/global/homes/z/zhan391/.conda/envs/e3sm_analysis/bin/python ../scripts/run_process_sst_index.py --sources obs e3sm smyle --outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/E3SMLE --smyle-outdir /global/cfs/cdirs/e3sm/S2S2D/s2d_diag/CESM-SMYLE --init-months 5 11 --year-start 1980 --year-end 2018 --climy0 1980 --climy1 2010 --nlead 24 --e3sm-nens 10 --smyle-nens 20 --workers 8 --regions IOD --custom-regions {"Nino12": {"lonlat": [270.0, 280.0, -10.0, 0.0], "long_name": "Nino 1+2 regional mean SST"}, "Nino3": {"lonlat": [210.0, 270.0, -5.0, 5.0], "long_name": "Nino 3 regional mean SST"}, "Nino3.4": {"lonlat": [190.0, 240.0, -5.0, 5.0], "long_name": "Nino 3.4 regional mean SST"}, "Nino4": {"lonlat": [160.0, 210.0, -5.0, 5.0], "long_name": "Nino 4 regional mean SST"}, "TNA": {"lonlat": [305.0, 345.0, 5.0, 25.0], "long_name": "TNA regional mean SST"}, "TSA": {"lonlat": [330.0, 10.0, -20.0, 0.0], "long_name": "TSA regional mean SST"}, "PACWRAMPOOL": {"l